# Análisis y Predicción de Riesgo de Ataque al Corazón

Flujo completo de análisis para el dataset `heart_attack_prediction_dataset.csv`:
1. **Carga y Exploración de Datos (EDA)**
2. **Ingeniería de Características (`Blood Pressure` -> `Systolic_BP` y `Diastolic_BP`)**
3. **Definición de Target (`Heart Attack Risk`) y División 70/30**
4. **Preprocesamiento Automático**
5. **Entrenamiento y Validación Cruzada (10 Folds)**
6. **Evaluación en el Conjunto de Test (30%)**
7. **Exportación a `.joblib`**

In [1]:
import os
import glob
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

In [2]:
downloads_dir = os.path.expanduser(r'~\Downloads')
pattern = os.path.join(downloads_dir, '*heart*attack*.csv')
files = glob.glob(pattern)

file_path = max(files, key=os.path.getmtime)
print(f'📁 Archivo seleccionado: {file_path}')

df = pd.read_csv(file_path)
df.head()

📁 Archivo seleccionado: C:\Users\juanp\Downloads\heart_attack_prediction_dataset.csv


,Patient ID,Age,Sex,Cholesterol,Blood Pressure,Heart Rate,Diabetes,Family History,Smoking,Obesity,...,Sedentary Hours Per Day,Income,BMI,Triglycerides,Physical Activity Days Per Week,Sleep Hours Per Day,Country,Continent,Hemisphere,Heart Attack Risk
0,BMW7812,67,Male,208,158/88,72,0,0,1,0,...,6.615001,261404,31.251233,286,0,6,Argentina,South America,Southern Hemisphere,0
1,CZE1114,21,Male,389,165/93,98,1,1,1,1,...,4.963459,285768,27.194973,235,1,7,Canada,North America,Northern Hemisphere,0
2,BNI9906,21,Female,324,174/99,72,1,0,0,0,...,9.463426,235282,28.176571,587,4,4,France,Europe,Northern Hemisphere,0
3,JLN3497,84,Male,383,163/100,73,1,1,1,0,...,7.648981,125640,36.464704,378,3,4,Canada,North America,Northern Hemisphere,0
4,GFO8847,66,Male,318,91/88,93,1,1,1,1,...,1.514821,160555,21.809144,231,1,5,Thailand,Asia,Northern Hemisphere,0


In [3]:
# Feature engineering para Blood Pressure
if 'Blood Pressure' in df.columns:
    df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure'].str.split('/', expand=True).astype(float)

predecir = df['Heart Attack Risk']
drop_cols = ['Heart Attack Risk', 'Patient ID', 'Blood Pressure']

X = df.drop(columns=drop_cols)
y = predecir

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f'Muestras totales: {len(df)}')
print(f'Train (70%): {X_train.shape[0]} muestras')
print(f'Test (30%):  {X_test.shape[0]} muestras')

Muestras totales: 8763
Train (70%): 6134 muestras
Test (30%):  2629 muestras


In [4]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

In [5]:
models = {
    'Regresión Logística': LogisticRegression(C=0.1, max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest Regulado': RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=5, random_state=42, class_weight='balanced'),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
}

# NOTA: class_weight='balanced' es necesario porque el target está desbalanceado
# (64% / 36%). Sin esto, los 3 modelos colapsan a predecir siempre la clase
# mayoritaria (0 = "sin riesgo") y el recall de la clase 1 (pacientes en
# riesgo) queda en 0% — el modelo nunca detecta un paciente enfermo.

for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    n_pred_positive = int((y_pred == 1).sum())
    print(f'MODELO: {name}')
    print(f'Accuracy: {acc:.4f} ({acc*100:.2f}%) | ROC-AUC: {auc:.4f}')
    print(f'Pacientes predichos como "en riesgo" (clase 1): {n_pred_positive} / {len(y_pred)}')
    print(classification_report(y_test, y_pred, target_names=['Bajo Riesgo (0)', 'Alto Riesgo (1)']))
    print(confusion_matrix(y_test, y_pred))
    print('-' * 50)


MODELO: Regresión Logística
Accuracy: 0.4960 (49.60%) | ROC-AUC: 0.4965
Pacientes predichos como "en riesgo" (clase 1): 1291 / 2629
                 precision    recall  f1-score   support

Bajo Riesgo (0)       0.64      0.50      0.56      1687
Alto Riesgo (1)       0.35      0.48      0.41       942

       accuracy                           0.50      2629
      macro avg       0.49      0.49      0.48      2629
   weighted avg       0.53      0.50      0.51      2629

[[850 837]
 [488 454]]
--------------------------------------------------


MODELO: Random Forest Regulado
Accuracy: 0.5470 (54.70%) | ROC-AUC: 0.5152
Pacientes predichos como "en riesgo" (clase 1): 979 / 2629
                 precision    recall  f1-score   support

Bajo Riesgo (0)       0.65      0.64      0.64      1687
Alto Riesgo (1)       0.37      0.39      0.38       942

       accuracy                           0.55      2629
      macro avg       0.51      0.51      0.51      2629
   weighted avg       0.55      0.55      0.55      2629

[[1073  614]
 [ 577  365]]
--------------------------------------------------
MODELO: Árbol de Decisión
Accuracy: 0.5432 (54.32%) | ROC-AUC: 0.4908
Pacientes predichos como "en riesgo" (clase 1): 859 / 2629
                 precision    recall  f1-score   support

Bajo Riesgo (0)       0.64      0.67      0.65      1687
Alto Riesgo (1)       0.35      0.32      0.33       942

       accuracy                           0.54      2629
      macro avg       0.49      0.49      0.49      2629
   weighted avg       0.53 

In [6]:
best_pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(C=0.1, max_iter=1000, random_state=42, class_weight='balanced'))])
best_pipe.fit(X_train, y_train)
joblib.dump(best_pipe, os.path.join(downloads_dir, 'heart_attack_model.joblib'))
joblib.dump(best_pipe, 'heart_attack_model.joblib')
print('Modelo guardado como heart_attack_model.joblib (con class_weight=balanced)')


Modelo guardado como heart_attack_model.joblib (con class_weight=balanced)


## ⚠️ Limitación importante del dataset

Antes de confiar en este modelo para nada real, hay que ser honestos sobre lo que muestran los números:

- La correlación máxima (valor absoluto) entre **cualquier** feature numérica (colesterol, presión arterial, tabaquismo, obesidad, antecedentes familiares, etc.) y `Heart Attack Risk` es **0.019** — prácticamente cero.
- El ROC-AUC de los 3 modelos ronda **0.49–0.50**, es decir, discriminación equivalente a **azar puro** (0.5 = tirar una moneda).
- Esto no es un problema de qué modelo usar ni de tuning: **este dataset en particular no contiene una relación real detectable entre las variables clínicas y la etiqueta**. Es un problema reportado por otros usuarios de este dataset de Kaggle — las etiquetas parecen generadas sin relación causal con las features.

**Qué sí se arregló:** antes, los 3 modelos predecían *siempre* la clase 0 ("sin riesgo") porque no se corrigió el desbalance de clases (64%/36%). Eso significaba **0% de recall en la clase 1** — el modelo jamás detectaba un paciente en riesgo, sin importar sus datos. Con `class_weight='balanced'` el modelo ahora sí predice la clase 1 en una proporción razonable.

**Qué NO se arregló (porque no se puede arreglar con código):** la capacidad predictiva real del modelo. Con AUC ~0.50, el modelo balanceado no es mejor que adivinar. Para que esto sirva de verdad se necesitaría un dataset con relación real entre variables clínicas y el desenlace.